# TrainLM on TPU

This is the same Hugging Face-like workflow an end user runs. The first cells clone and refresh the requested TrainLM branch, then install the package and dependencies. Choose a TPU runtime, set the shard count and local cache/output directories below, then call `trainer.train()`. This smoke uses the pretrained HuggingFaceTB/SmolLM2-135M-Instruct checkpoint so model preflight fits comfortably on a v5e-8. TrainLM resolves the dataset revision to an immutable commit before launching workers.

TrainLM handles TPU discovery, world size, worker launch, ranks, preflight, caching, distributed data ownership, checkpoints, evaluation, and structured results under the hood.


In [ ]:
!git clone --branch milestone/m10-m12-kernels-parity --single-branch https://github.com/Dhiraj309/TrainLM.git


In [ ]:
%cd TrainLM


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/TrainLM

BRANCH="milestone/m10-m12-kernels-parity"

git fetch origin "$BRANCH"
git checkout -B "$BRANCH" "origin/$BRANCH"

echo "Current TrainLM revision:"
git log -1 --oneline
git rev-parse HEAD


## 1. Install and restart once

Kaggle preinstalls TensorFlow and mismatched vision/audio wheels that can initialize or conflict with Torch/XLA in the notebook process. TrainLM text pretraining does not use them. Run the cleanup/install cell, wait for it to finish, then use **Restart Session** before running section 2. Do not continue in the same kernel.


In [ ]:
%pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu tensorflow-intel tensorflow-rocm tf-keras keras torchvision torchaudio
%pip install -e ".[tpu-xla]" -c constraints/tpu-xla-2.9.txt

# Some Kaggle images leave an orphaned tensorflow/ package directory after
# its distribution metadata is removed. Remove only that exact package
# directory; the required restart below clears any already-loaded module.
import importlib.util
from pathlib import Path
import shutil
_tensorflow_spec = importlib.util.find_spec("tensorflow")
if _tensorflow_spec is not None and _tensorflow_spec.submodule_search_locations:
    for _location in _tensorflow_spec.submodule_search_locations:
        _path = Path(_location)
        if _path.name == "tensorflow" and "site-packages" in _path.parts:
            print("Removing orphan TensorFlow package directory:", _path)
            shutil.rmtree(_path, ignore_errors=False)


## 2. Your inputs

Choose how many numbered shards to use. `(0, TRAIN_SHARD_STOP)` is end-exclusive, so `2` downloads shards `00000` and `00001`. The next shard is reserved for evaluation. TrainLM resolves `main` to its immutable Hub commit and saves the files in `DATA_CACHE_DIR`.

The validation model is the pretrained HuggingFaceTB/SmolLM2-135M-Instruct checkpoint. It is a dense Llama-family decoder with GQA (9 query heads / 3 KV heads) and a 49,152-token vocabulary. The supplied LaughLM binary shards are retained for a lifecycle smoke only; they were tokenized with a different vocabulary, so do not interpret loss or quality as SmolLM2 training evidence until SmolLM2-tokenized data is used.


In [ ]:
import importlib.util
from pathlib import Path

_tensorflow_spec = importlib.util.find_spec("tensorflow")
if _tensorflow_spec is not None:
    raise RuntimeError(
        "TensorFlow is still importable after cleanup. Restart the Kaggle session after section 1; "
        f"origin={_tensorflow_spec.origin!r}, locations={_tensorflow_spec.submodule_search_locations!r}"
    )

MODEL_CONFIG = {
    "provider": "huggingface",
    "initialization": "pretrained",
    "name_or_path": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "revision": "main",
    "dtype": "bfloat16",
    "trust_remote_code": False,
}
DATASET_ID = "LaughTaleAI/LaughLM-Tokenized-Fine"
DATASET_REVISION = "main"
TRAIN_SHARD_STOP = 1
DATA_CACHE_DIR = Path("/kaggle/working/huggingface-cache")
OUTPUT_DIR = Path("/kaggle/working/trainlm-smollm2-135m-smoke")


## 3. Build the datasets and trainer

`from_hub()` downloads the requested `.bin` range once into `DATA_CACHE_DIR`, scans and validates it before TPU launch, then trains from those local files through lazy memory maps. It does not download during training. Users do not write download loops or configure a dataloader, process count, rank, or world size.


In [ ]:
from trainlm import PackedBinDataset, TrainLMTrainer

# Keep the lifecycle smoke compilation-small; raise only after this path passes.
sequence_length = 128
train_dataset = PackedBinDataset.from_hub(
    DATASET_ID,
    revision=DATASET_REVISION,
    shard_range=(0, TRAIN_SHARD_STOP),
    sequence_length=sequence_length,
    cache_dir=DATA_CACHE_DIR,
    split="train",
)
eval_dataset = PackedBinDataset.from_hub(
    DATASET_ID,
    revision=train_dataset.hub_revision,
    shard_range=(TRAIN_SHARD_STOP, TRAIN_SHARD_STOP + 1),
    sequence_length=sequence_length,
    cache_dir=DATA_CACHE_DIR,
    split="validation",
)

trainer = TrainLMTrainer.from_config(
    {
        "api_version": "1",
        "model": MODEL_CONFIG,
        "training_args": {
            "output_dir": OUTPUT_DIR,
            "accelerator": "tpu",
            "bf16": True,
            "max_steps": 6,
            "sequence_length": sequence_length,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 1,
            "logging_steps": 1,
            "eval_steps": 2,
            "save_steps": 2,
        },
    },
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)


## 4. Train

This single call performs the private collective probe and model preflight, launches every available TPU worker, trains, evaluates every two steps, and writes committed checkpoints every two steps. If the cell is interrupted or a worker fails, TrainLM terminates the complete private worker process group before returning the error, so you can fix the input and rerun. If the failed run already committed a checkpoint, resume it or choose a new output directory instead of overwriting it.

In [ ]:
result = trainer.train()
result

## 5. Optional: inspect what TrainLM selected

`explain()` is useful when reviewing fallbacks or filing a result. It does not require users to inspect worker commands or logs.

In [ ]:
trainer.explain(format="text")

## 6. Optional: resume

Normally set this to the last committed checkpoint after an interrupted or completed run. TrainLM validates topology and restores model, optimizer, scheduler, runtime, RNG, trainer, and packed-data position internally.

In [ ]:
resumed_trainer = TrainLMTrainer.from_config(
    {
        "api_version": "1",
        "model": MODEL_CONFIG,
        "training_args": {
            "output_dir": "/kaggle/working/trainlm-smollm2-135m-resumed",
            "accelerator": "tpu",
            "bf16": True,
            "max_steps": 10,
            "sequence_length": sequence_length,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 1,
            "logging_steps": 1,
            "eval_steps": 2,
            "save_steps": 2,
        },
    },
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
resumed_result = resumed_trainer.train(
    resume_from_checkpoint=OUTPUT_DIR / "checkpoint-4"
)
resumed_result


## What to save from a validation run

Archive the output directory, including `coordinator_summary.json`, `summary.json`, `metrics.jsonl`, committed checkpoint manifests/shards, and XLA metrics. The returned result is the normal user-facing status; these files are only needed for debugging or performance certification.

A successful run validates the lifecycle on that TPU. It does not by itself mark performance as certified.